# Body features (YOLO person crops, CLIP ViT-B/32) for Hi-EF or MELD

The Hi-EF paper's visual branches are face, whole frame (scene) and **person (body)**. The released features
(`hi-ef-features-v2`) hold face and whole-frame CLIP features only, so this notebook adds the missing branch:
16 frames per clip (evenly spaced, as in `meld_prepare.ipynb`), the largest detected person in each frame (ultralytics YOLO, class
*person*), CLIP ViT-B/32 image features of the crop (the same encoder as the face and scene features).
Only clips I–III are processed; clip IV is never read. Set `SOURCE = 'hief'` (Hi-EF clips) or `'meld'` (MELD clips;
attach the output of `meld_prepare.ipynb` and, optionally, a MELD.Raw dataset via `MELD_LOCAL`).

Output: `body_<SOURCE>.pt` = `{clip_id: (features [16, 512] fp16, valid mask [16] bool)}`.
Frames without a detected person have mask False and zero features. The detector is not the authors' (their script
is not in the repository), so the crops are similar in kind, not identical.

In [ ]:
!pip install -q ultralytics

In [ ]:
# ======== CONFIG ========
import os, glob
SOURCE = "hief"                                   # "hief" or "meld"
OUT_DIR = "/kaggle/working"
N_FRAMES = 16
YOLO_WEIGHTS = "yolov8s.pt"                       # downloaded by ultralytics (internet on)
PERSON_CONF, MIN_PERSON_H = 0.4, 64               # confidence and minimum crop height in pixels
CROP_MARGIN = 0.05
DEBUG_N = None                                    # e.g. 40 clips for a smoke test

DATASET_DIR = "/kaggle/input/datasets/USER/hi-ef-dataset"
SPLIT_CSV = "/kaggle/input/datasets/USER/hi-ef-split/source_folder_split_seed42.csv"
MELD_LOCAL = None                                 # MELD.Raw dataset folder; None = auto-detect an attached MELD dataset


def find_meld_raw():
    # an attached dataset whose folder name contains "meld", other than the output of meld_prepare (meld-m1 / meld_hief / meld_features)
    out = []
    for base in ["/kaggle/input"] + sorted(glob.glob("/kaggle/input/datasets/*")):
        for d in sorted(glob.glob(base + "/*")):
            n = os.path.basename(d).lower()
            if os.path.isdir(d) and "meld" in n and "m1" not in n and "hief" not in n and "features" not in n and "probs" not in n:
                has = next(glob.iglob(d + "/**/*.mp4", recursive=True), None) or next(glob.iglob(d + "/**/*.tar.gz", recursive=True), None)
                if has:
                    out.append(d)
    return out


if SOURCE == "meld" and MELD_LOCAL is None:
    _c = find_meld_raw()
    assert _c, "attach the MELD.Raw dataset (the Kaggle 'Multimodal EmotionLines Dataset (MELD)') or set MELD_LOCAL"
    MELD_LOCAL = _c[0]
    print("MELD_LOCAL =", MELD_LOCAL)
WORK = "/tmp/meld"


def find_one(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"attach the dataset holding {pattern}")
    return hits[0]

In [ ]:
import numpy as np, pandas as pd
import os, re, glob, subprocess
from concurrent.futures import ThreadPoolExecutor

MELD_URL = "https://web.eecs.umich.edu/~mihalcea/downloads/MELD.Raw.tar.gz"
CSV_URL = "https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/{}_sent_emo.csv"
EMO_MAP = {'anger': 'angry', 'disgust': 'disgust', 'fear': 'fear', 'joy': 'happy', 'neutral': 'neutral',
           'sadness': 'sad', 'surprise': 'surprise'}
SPLITS = (('tr', 'train', 'train'), ('dv', 'dev', 'val'), ('te', 'test', 'test'))   # prefix, MELD name, Hi-EF name


MELD_HELP = ("Could not download MELD.Raw automatically. Attach a Kaggle dataset that holds MELD.Raw "
             "(Add Input -> search 'MELD') and set MELD_LOCAL to its folder, e.g. MELD_LOCAL = '/kaggle/input/<name>'. "
             "Unextracted train/dev/test .tar.gz files inside it are fine.")


def _probe_gzip(url):
    # first two bytes of the response must be the gzip magic 1f 8b; prints the real error otherwise
    pr = subprocess.Popen(['curl', '-fsSL', '--max-time', '60', url], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    head = pr.stdout.read(2)
    pr.kill()
    err = pr.stderr.read().decode(errors='replace').strip()
    pr.wait()
    if head != b'\x1f\x8b':
        print(f"  {url}: {err[:300]!r}, first bytes {head!r}", flush=True)
        return False
    return True


def _unpack_inner(src_root, dst_root):
    # MELD.Raw ships train/dev/test as inner .tar.gz archives; unpack them into dst_root (writable)
    for inner in glob.glob(f"{src_root}/**/*.tar.gz", recursive=True):
        rel = os.path.dirname(os.path.relpath(inner, src_root))
        out = os.path.join(dst_root, rel)
        os.makedirs(out, exist_ok=True)
        print("  unpacking", inner, flush=True)
        subprocess.run(['tar', '-xzf', inner, '-C', out], check=True)
        if inner.startswith(dst_root):
            os.remove(inner)


def fetch_meld():
    # returns {(prefix, dialogue, utterance): mp4 path} and {prefix: csv path}
    root = MELD_LOCAL
    if root:
        if not glob.glob(f"{root}/**/*.mp4", recursive=True):
            os.makedirs(WORK, exist_ok=True)
            if not glob.glob(f"{WORK}/**/*.mp4", recursive=True):
                _unpack_inner(root, WORK)            # read-only input: unpack the inner archives into WORK
            assert glob.glob(f"{WORK}/**/*.mp4", recursive=True), f"no MELD .mp4 or .tar.gz found under {root}"
            for c in glob.glob(f"{root}/**/*_sent_emo.csv", recursive=True):
                if not os.path.exists(os.path.join(WORK, os.path.basename(c))):
                    subprocess.run(['cp', c, WORK], check=True)
            root = WORK
    else:
        root = WORK
        os.makedirs(WORK, exist_ok=True)
        if not glob.glob(f"{WORK}/**/*.mp4", recursive=True):
            print("checking the MELD.Raw download link ...", flush=True)
            if not _probe_gzip(MELD_URL):
                raise RuntimeError(MELD_HELP)
            print("downloading and unpacking MELD.Raw (about 10 GB) ...", flush=True)
            subprocess.run(f"set -o pipefail; curl -fsSL --retry 3 {MELD_URL} | tar -xz -C {WORK}", shell=True,
                           check=True, executable='/bin/bash')
            _unpack_inner(WORK, WORK)
    vids = {}
    for p in glob.glob(f"{root}/**/*.mp4", recursive=True):
        m = re.fullmatch(r'dia(\d+)_utt(\d+)\.mp4', os.path.basename(p))
        if not m:
            continue                                   # skips macOS '._' files and other names
        low = os.path.relpath(p, root).lower()
        s = 'te' if 'test' in low else 'dv' if 'dev' in low else 'tr' if 'train' in low else None
        if s:
            vids[(s, int(m[1]), int(m[2]))] = p
    csvs = {}
    for s, name, _ in SPLITS:
        found = glob.glob(f"{root}/**/{name}_sent_emo.csv", recursive=True)
        if found:
            csvs[s] = found[0]
        else:
            csvs[s] = f"{WORK}/{name}_sent_emo.csv"
            os.makedirs(WORK, exist_ok=True)
            subprocess.run(['curl', '-fsSL', '--retry', '3', '-o', csvs[s], CSV_URL.format(name)], check=True)
    print("videos found per split:", {s: sum(k[0] == s for k in vids) for s, _, _ in SPLITS})
    return vids, csvs


def clip_id(s, d, u):
    return f"{s}{int(d):05d}/{int(u):03d}"


def extract_wavs(clips, vids, sr):
    # mono wav per clip at <WORK>/audio/<ep>/<num>.wav (ffmpeg), skipped when present
    def one(c):
        s, d, u = c[:2], int(c[2:7]), int(c[8:])
        out = f"{WORK}/audio/{c}.wav"
        if os.path.exists(out) or (s, d, u) not in vids:
            return
        os.makedirs(os.path.dirname(out), exist_ok=True)
        subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', vids[(s, d, u)], '-vn', '-ac', '1', '-ar', str(sr),
                        out], check=False)
    with ThreadPoolExecutor(8) as pool:
        list(pool.map(one, clips))
    print("wav files:", len(glob.glob(f"{WORK}/audio/**/*.wav", recursive=True)), flush=True)

import numpy as np, pandas as pd

In [ ]:
import time, torch, cv2
from PIL import Image
if SOURCE == "hief":
    roots = sorted(glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF")))
    VIDEO_ROOTS = [os.path.join(r, "video") for r in roots if os.path.isdir(os.path.join(r, "video"))]

    def video_of(c):
        ep, num = c.split('/')
        for root in VIDEO_ROOTS:
            p = os.path.join(root, ep, num + ".mp4")
            if os.path.exists(p):
                return p
        return None
    sp = pd.read_csv(SPLIT_CSV, dtype=str, usecols=['sample_id', 'split', 'clip1', 'clip2', 'clip3'])
else:
    vids, csvs = fetch_meld()
    sp = pd.read_csv(find_one("meld_split.csv"), dtype=str)

    def video_of(c):
        return vids.get((c[:2], int(c[2:7]), int(c[8:])))
CLIPS = sorted(set(sp[['clip1', 'clip2', 'clip3']].values.ravel()))
if DEBUG_N:
    CLIPS = CLIPS[:DEBUG_N]
print(f"{SOURCE}: clips I-III {len(CLIPS)} | without video {sum(video_of(c) is None for c in CLIPS)}")

In [ ]:
from ultralytics import YOLO
from transformers import CLIPModel, CLIPProcessor
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
yolo = YOLO(YOLO_WEIGHTS)
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
feat = lambda o: (o if torch.is_tensor(o) else o.pooler_output).float().cpu()


def frames_of(path):
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    keep = set(np.linspace(0, max(n - 1, 0), N_FRAMES).astype(int).tolist()) if n else set()
    out, i = [], 0
    while len(out) < len(keep):
        ok, fr = cap.read()
        if not ok:
            break
        if i in keep:
            out.append(fr)
        i += 1
    cap.release()
    return out


def person_crops(frs):
    """largest person (class 0) per frame, or None"""
    res = yolo(frs, classes=[0], conf=PERSON_CONF, verbose=False, device=0 if DEVICE == "cuda" else "cpu")
    crops = []
    for fr, r in zip(frs, res):
        H, W = fr.shape[:2]
        b = r.boxes.xyxy.cpu().numpy() if r.boxes is not None and len(r.boxes) else np.zeros((0, 4))
        b = [x for x in b if x[3] - x[1] >= MIN_PERSON_H]
        if not b:
            crops.append(None)
            continue
        x1, y1, x2, y2 = max(b, key=lambda x: (x[2] - x[0]) * (x[3] - x[1]))
        mw, mh = CROP_MARGIN * (x2 - x1), CROP_MARGIN * (y2 - y1)
        crops.append(fr[int(max(0, y1 - mh)):int(min(H, y2 + mh)), int(max(0, x1 - mw)):int(min(W, x2 + mw))])
    return crops

In [ ]:
OUT = f"{OUT_DIR}/body_{SOURCE}.pt"
BODY = torch.load(OUT, map_location="cpu") if os.path.exists(OUT) else {}
todo = [c for c in CLIPS if c not in BODY]
print(f"to extract: {len(todo)} / {len(CLIPS)}")
stats = {"no_video": 0, "frames": 0, "body_frames": 0}
t0 = time.time()
for n, c in enumerate(todo):
    p = video_of(c)
    frs = frames_of(p) if p else []
    stats["no_video"] += not frs
    f = torch.zeros(N_FRAMES, 512); m = torch.zeros(N_FRAMES, dtype=torch.bool)
    if frs:
        crops = person_crops(frs)
        ok = [k for k, cr in enumerate(crops) if cr is not None and cr.size]
        if ok:
            imgs = [Image.fromarray(cv2.cvtColor(crops[k], cv2.COLOR_BGR2RGB)) for k in ok]
            with torch.no_grad():
                o = feat(clip.get_image_features(pixel_values=proc(images=imgs, return_tensors="pt").pixel_values.to(DEVICE)))
            for j, k in enumerate(ok):
                f[k], m[k] = o[j], True
        stats["frames"] += len(frs); stats["body_frames"] += int(m.sum())
    BODY[c] = (f.half(), m)
    if (n + 1) % 500 == 0:
        torch.save(BODY, OUT)
        el = (time.time() - t0) / 60
        print(f"{n + 1}/{len(todo)} | {el:.1f} min | ETA {el / (n + 1) * (len(todo) - n - 1):.0f} min | {stats}", flush=True)
torch.save(BODY, OUT)
print("done", stats)

## Sanity checks (no labels)

In [ ]:
valid = torch.stack([BODY[c][1] for c in CLIPS]).float()
print(f"clips with at least one body frame: {(valid.sum(1) > 0).float().mean() * 100:.1f}% | "
      f"mean valid frames per clip {valid.sum(1).mean():.1f} / {N_FRAMES}")